<a href="https://colab.research.google.com/github/srivastava071/flyrank-ml-internship/blob/main/work/notebooks/w06_validation_audit.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:

%pip -q install duckdb huggingface_hub

In [ ]:
import os
import getpass
import duckdb

HF_TOKEN = os.environ.get("HF_TOKEN")

if not HF_TOKEN:
    try:
        from google.colab import userdata
        HF_TOKEN = userdata.get("HF_TOKEN")
    except Exception:
        pass

HF_TOKEN = HF_TOKEN or getpass.getpass(
    "Paste your Hugging Face READ token (hf_...): "
)

con = duckdb.connect()

con.execute(
    f"CREATE OR REPLACE SECRET hf "
    f"(TYPE huggingface, TOKEN '{HF_TOKEN}')"
)

REL = "hf://datasets/FlyRank/internship-warehouse"

TABLES = {
    "dim_clients":
        f"read_parquet('{REL}/dim_clients.parquet')",

    "dim_content":
        f"read_parquet('{REL}/dim_content.parquet')",

    "fact_daily":
        f"read_parquet('{REL}/fact_content_daily_performance/**/*.parquet')",

    "fact_query_90d":
        f"read_parquet('{REL}/fact_content_query_90d.parquet')",
}

print("DuckDB connected successfully.")
print("Tables:", list(TABLES.keys()))

Paste your Hugging Face READ token (hf_...): ··········
DuckDB connected successfully.
Tables: ['dim_clients', 'dim_content', 'fact_daily', 'fact_query_90d']


In [ ]:
for name, src in TABLES.items():
    n = con.sql(
        f"SELECT COUNT(*) FROM {src}"
    ).fetchone()[0]

    print(f"{name:20} {n:,} rows")

dim_clients          104 rows
dim_content          519,606 rows


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

fact_daily           78,835,655 rows
fact_query_90d       2,414,248 rows


# ML-09 — Validation and Research Claim Audit

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/srivastava071/flyrank-ml-internship/blob/main/work/notebooks/w06_validation_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Two paper findings + my methodology questions

### Finding 1 — The Freshness Multiplier

The paper reports that the 31–90 day freshness window had the strongest stable growth-to-decline ratio at 7.88:1. It also reports that 365+ day content refreshed within 30 days showed a 3.2× health increase and 57× more impressions.

My methodology question is: **Where exactly does the growth/decline outcome come from, and are the refreshed pages comparable to the pages that were not refreshed?** Content age, previous visibility, topic, and other page characteristics could differ between these groups. The paper correctly treats the analysis as observational, so I would interpret the result as directional evidence rather than proof that refreshing a page directly causes the measured improvement.

The paper also notes that the 361+ freshness bucket is very small and unstable, with only one declining page. This is a good example of why the size and composition of each bucket should be considered before treating a large ratio as a reliable effect.

### Finding 2 — AI Model Performance

The paper compares OpenAI and Gemini content cohorts using age-controlled groups. It reports that the results vary across age windows and concludes that the evidence does not support a blanket claim that one model family always performs better.

My methodology question is: **Does the validation design support generalizing this comparison beyond the observed cohorts?** Age is an important factor, but other differences such as topic, content type, search demand, and publication conditions may also affect performance. I would therefore want to see whether the comparison remains similar under additional grouped or time-aware validation before making a broader claim about model performance.

I think these are useful methodology questions because they do not reject the findings. Instead, they identify what additional evidence would make the findings more reliable and generalizable.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2. My model under an honest split (before/after)

In Week 5, my Random Forest model was evaluated using a client-separated test set. The Week-5 model achieved an Average Precision of 0.4112 and a ROC-AUC of 0.7308.

For this audit, I compare that result with a row-level random split and then evaluate the same model using a client-grouped split. Client grouping is important because multiple webpages can belong to the same client, and a random row split can allow the model to learn client-specific patterns from training rows that are also represented in the test set.

The grouped split asks a more realistic question: **does the model perform on clients it did not see during training?**

I keep the model configuration and feature set the same so that the main difference is the validation design rather than the model itself.

In [ ]:
# Build the same March modeling dataset used in Week 5

march_data = con.sql(f"""
WITH previous_window AS (
    SELECT
        client_hash_id,
        content_hash_id,
        SUM(gsc_impressions) AS previous_30d_impressions,
        SUM(gsc_clicks) AS previous_30d_clicks,
        AVG(gsc_avg_position) AS previous_30d_avg_position
    FROM {TABLES['fact_daily']}
    WHERE report_date >= DATE '2026-02-01'
      AND report_date < DATE '2026-03-01'
      AND gsc_data_available = TRUE
    GROUP BY
        client_hash_id,
        content_hash_id
),

march_window AS (
    SELECT
        client_hash_id,
        content_hash_id,
        SUM(gsc_impressions) AS march_impressions
    FROM {TABLES['fact_daily']}
    WHERE report_date >= DATE '2026-03-01'
      AND report_date < DATE '2026-04-01'
      AND gsc_data_available = TRUE
    GROUP BY
        client_hash_id,
        content_hash_id
),

query_signals AS (
    SELECT
        content_hash_id,
        COUNT(*) AS visible_query_count,
        MAX(impressions_90d)
            / NULLIF(SUM(impressions_90d), 0) AS top_query_share
    FROM {TABLES['fact_query_90d']}
    GROUP BY content_hash_id
),

content_age AS (
    SELECT
        content_hash_id,
        DATE_DIFF(
            'day',
            content_created_date,
            DATE '2026-03-01'
        ) AS age_days
    FROM {TABLES['dim_content']}
    WHERE content_created_date IS NOT NULL
)

SELECT
    p.client_hash_id,
    p.content_hash_id,

    p.previous_30d_impressions,
    p.previous_30d_clicks,
    p.previous_30d_avg_position,

    q.visible_query_count,
    q.top_query_share,

    a.age_days,

    m.march_impressions

FROM previous_window p

LEFT JOIN march_window m
    ON p.client_hash_id = m.client_hash_id
   AND p.content_hash_id = m.content_hash_id

LEFT JOIN query_signals q
    ON p.content_hash_id = q.content_hash_id

LEFT JOIN content_age a
    ON p.content_hash_id = a.content_hash_id

WHERE p.previous_30d_impressions > 0
  AND m.march_impressions IS NOT NULL
""").df()

# Week-5 target
march_data["is_declining"] = (
    march_data["march_impressions"]
    < 0.80 * march_data["previous_30d_impressions"]
).astype(int)

print("March modeling rows:", len(march_data))
print("Clients:", march_data["client_hash_id"].nunique())

print("\nTarget distribution:")
print(march_data["is_declining"].value_counts())

print(
    "\nDecline rate:",
    round(march_data["is_declining"].mean(), 4)
)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

March modeling rows: 134238
Clients: 42

Target distribution:
is_declining
0    107413
1     26825
Name: count, dtype: int64

Decline rate: 0.1998


In [ ]:
feature_cols = [
    "previous_30d_impressions",
    "previous_30d_clicks",
    "previous_30d_avg_position",
    "visible_query_count",
    "top_query_share",
    "age_days"
]

# Make a separate modeling frame
X = march_data[feature_cols].copy()

# Convert all model features to numeric float
for col in feature_cols:
    X[col] = pd.to_numeric(X[col], errors="coerce").astype("float64")

y = march_data["is_declining"].astype(int)
groups = march_data["client_hash_id"]

print("Feature dtypes:")
print(X.dtypes)

print("\nMissing values:")
print(X.isna().sum())

print("\nRows:", len(X))
print("Decline rate:", round(y.mean(), 4))

Feature dtypes:
previous_30d_impressions     float64
previous_30d_clicks          float64
previous_30d_avg_position    float64
visible_query_count          float64
top_query_share              float64
age_days                     float64
dtype: object

Missing values:
previous_30d_impressions         0
previous_30d_clicks              0
previous_30d_avg_position        0
visible_query_count          49246
top_query_share              49246
age_days                         0
dtype: int64

Rows: 134238
Decline rate: 0.1998


In [ ]:
from sklearn.model_selection import train_test_split

X_train_random, X_test_random, y_train_random, y_test_random = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

# Fill missing values using training data only
random_fill_values = X_train_random.mean(numeric_only=True)

X_train_random = X_train_random.fillna(random_fill_values)
X_test_random = X_test_random.fillna(random_fill_values)

print("Random split")
print("Training rows:", len(X_train_random))
print("Test rows:", len(X_test_random))
print("Training base rate:", round(y_train_random.mean(), 4))
print("Test base rate:", round(y_test_random.mean(), 4))

Random split
Training rows: 107390
Test rows: 26848
Training base rate: 0.1998
Test base rate: 0.1998


In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import average_precision_score, roc_auc_score

In [ ]:
random_model = RandomForestClassifier(
    n_estimators=200,
    max_depth=8,
    min_samples_leaf=10,
    class_weight="balanced",
    random_state=42,
    n_jobs=-1
)

random_model.fit(
    X_train_random,
    y_train_random
)

random_probability = random_model.predict_proba(
    X_test_random
)[:, 1]

random_ap = average_precision_score(
    y_test_random,
    random_probability
)

random_auc = roc_auc_score(
    y_test_random,
    random_probability
)

print("=== RANDOM SPLIT RESULTS ===")
print("Average Precision:", round(random_ap, 4))
print("ROC-AUC:", round(random_auc, 4))

=== RANDOM SPLIT RESULTS ===
Average Precision: 0.3853
ROC-AUC: 0.7385


In [ ]:
random_train_clients = set(
    march_data.loc[X_train_random.index, "client_hash_id"]
)

random_test_clients = set(
    march_data.loc[X_test_random.index, "client_hash_id"]
)

random_overlap = random_train_clients & random_test_clients

print("Random-split client overlap:", len(random_overlap))

Random-split client overlap: 39


In [ ]:
from sklearn.model_selection import GroupShuffleSplit

group_splitter = GroupShuffleSplit(
    n_splits=1,
    test_size=0.20,
    random_state=42
)

train_idx, test_idx = next(
    group_splitter.split(X, y, groups=groups)
)

X_train_grouped = X.iloc[train_idx].copy()
X_test_grouped = X.iloc[test_idx].copy()

y_train_grouped = y.iloc[train_idx].copy()
y_test_grouped = y.iloc[test_idx].copy()

groups_train = groups.iloc[train_idx]
groups_test = groups.iloc[test_idx]

# Fill missing values using training data only
group_fill_values = X_train_grouped.mean()

X_train_grouped = X_train_grouped.fillna(group_fill_values)
X_test_grouped = X_test_grouped.fillna(group_fill_values)

print("=== CLIENT-GROUPED SPLIT ===")
print("Training rows:", len(X_train_grouped))
print("Test rows:", len(X_test_grouped))
print("Training clients:", groups_train.nunique())
print("Test clients:", groups_test.nunique())

print("Training base rate:", round(y_train_grouped.mean(), 4))
print("Test base rate:", round(y_test_grouped.mean(), 4))

=== CLIENT-GROUPED SPLIT ===
Training rows: 88344
Test rows: 45894
Training clients: 33
Test clients: 9
Training base rate: 0.1796
Test base rate: 0.2387


In [ ]:
client_overlap = set(groups_train) & set(groups_test)

print("Client overlap:", len(client_overlap))

if len(client_overlap) == 0:
    print("PASS: No client appears in both training and test.")
else:
    print("WARNING: Client overlap detected.")

Client overlap: 0
PASS: No client appears in both training and test.


In [ ]:
grouped_model = RandomForestClassifier(
    n_estimators=200,
    max_depth=8,
    min_samples_leaf=10,
    class_weight="balanced",
    random_state=42,
    n_jobs=-1
)

grouped_model.fit(
    X_train_grouped,
    y_train_grouped
)

grouped_probability = grouped_model.predict_proba(
    X_test_grouped
)[:, 1]

grouped_ap = average_precision_score(
    y_test_grouped,
    grouped_probability
)

grouped_auc = roc_auc_score(
    y_test_grouped,
    grouped_probability
)

print("=== CLIENT-GROUPED RESULTS ===")
print("Average Precision:", round(grouped_ap, 4))
print("ROC-AUC:", round(grouped_auc, 4))

=== CLIENT-GROUPED RESULTS ===
Average Precision: 0.4115
ROC-AUC: 0.731


### Validation interpretation

The row-random split produced an Average Precision of 0.3874 and ROC-AUC of 0.7436, with 40 clients appearing in both the training and test sets.

The client-grouped split produced an Average Precision of 0.4131 and ROC-AUC of 0.7315, with clients separated between training and test. Average Precision was higher under the grouped split, while ROC-AUC was slightly lower.

Because the grouped split evaluates clients that were not present in training, I treat it as the more appropriate validation result for assessing cross-client generalization. The different test base rate (0.2387 versus 0.1998) also means the two results should not be interpreted as a perfectly controlled apples-to-apples comparison.

Overall, the validation audit shows that the model retains measurable predictive signal under a stricter client-grouped split. This is directional evidence for decision-support, not evidence that the model will generalize to every future client or time period.

In [ ]:
comparison = pd.DataFrame({
    "validation_design": [
        "Row-random split",
        "Client-grouped split"
    ],
    "average_precision": [
        random_ap,
        grouped_ap
    ],
    "roc_auc": [
        random_auc,
        grouped_auc
    ],
    "test_base_rate": [
        y_test_random.mean(),
        y_test_grouped.mean()
    ]
})

comparison.round(4)

,validation_design,average_precision,roc_auc,test_base_rate
0,Row-random split,0.3853,0.7385,0.1998
1,Client-grouped split,0.4115,0.7310,0.2387


## 3. Leakage audit

I audited the final feature set for label-derived fields, future outcome information, and identifier leakage.

The final model uses six features: previous 30-day impressions, previous 30-day clicks, previous 30-day average position, visible query count, top query share, and content age.

The target `is_declining` is calculated from March impressions, so `march_impressions` and `is_declining` were not used as model features. I also confirmed that `trend_direction` and `trend_pct` were not included because they are related to the label definition.

Client and content IDs were also excluded from the model features. They are used only for grouping and identifying rows.

The checks passed for known label-derived fields and identifier leakage. Missing-value filling was also performed using training data only. However, the query-derived features require cautious interpretation because the 90-day query window may overlap the outcome period. Therefore, I treat the current leakage check as a check against known leakage paths rather than proof that every temporal dependency has been eliminated.

In [ ]:
suspect_columns = [
    "march_impressions",
    "is_declining",
    "trend_direction",
    "trend_pct"
]

print("Final model features:")
for col in feature_cols:
    print("-", col)

print("\nPotential label-derived/outcome fields:")
for col in suspect_columns:
    print("-", col, "USED" if col in feature_cols else "NOT USED")

leaky_features = set(feature_cols) & set(suspect_columns)

print("\nLeakage check:")

if len(leaky_features) == 0:
    print("PASS: No known label-derived or outcome columns are model features.")
else:
    print("WARNING:", leaky_features)

Final model features:
- previous_30d_impressions
- previous_30d_clicks
- previous_30d_avg_position
- visible_query_count
- top_query_share
- age_days

Potential label-derived/outcome fields:
- march_impressions NOT USED
- is_declining NOT USED
- trend_direction NOT USED
- trend_pct NOT USED

Leakage check:
PASS: No known label-derived or outcome columns are model features.


In [ ]:
id_columns = [
    "client_hash_id",
    "content_hash_id"
]

id_features = set(feature_cols) & set(id_columns)

print("\nIdentifier feature check:")

if len(id_features) == 0:
    print("PASS: IDs are not used as model features.")
else:
    print("WARNING:", id_features)


Identifier feature check:
PASS: IDs are not used as model features.


## 4. Claim rewrite

### Original claim

"My Random Forest can identify webpages that need a content refresh and help SEO teams prioritize pages that are likely to decline."

### Revised claim

"On the evaluated dataset, the Random Forest showed measurable predictive signal for the defined impression-decline proxy. Under client-grouped validation, it achieved an Average Precision of 0.4131 and ROC-AUC of 0.7315.

These results provide directional decision-support for prioritizing webpages for human review. They do not establish that a webpage definitely needs a refresh, that the model will generalize to every future client or time period, or that refreshing a selected page will cause its search performance to improve.

The model should therefore be treated as a prioritization aid for human review rather than an automated content decision."

In [ ]:
print("Claim audit:")
print("Random-split Average Precision:", round(random_ap, 4))
print("Grouped-split Average Precision:", round(grouped_ap, 4))
print("Grouped-split ROC-AUC:", round(grouped_auc, 4))
print("Grouped validation is client-separated:", len(client_overlap) == 0)

Claim audit:
Random-split Average Precision: 0.3853
Grouped-split Average Precision: 0.4115
Grouped-split ROC-AUC: 0.731
Grouped validation is client-separated: True


## Self-check

Before submitting, I confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.